# 🎧 Customer Support Agent System — Groq + `openai/gpt-oss-120b` (Colab Edition)

This notebook adapts the original course project to run **entirely for free in Google Colab**, using:

- **[openai-agents](https://github.com/openai/openai-agents-python)** SDK (the same `Agent` / `Runner` / handoff / guardrail primitives as the original code)
- **Groq's OpenAI-compatible API** as the backend, serving the open-source **`openai/gpt-oss-120b`** model instead of a paid OpenAI model
- A custom **DuckDuckGo web-search tool** in place of OpenAI's hosted `WebSearchTool` (which only works with OpenAI's own backend)

**Architecture (unchanged):**

```
User → Triage Agent → Order Status Agent (lookup_order tool)
                     → Refund Agent (process_refund tool)
                     → FAQ Agent (web search tool)
```

### Before you run this
Get a **free Groq API key** from https://console.groq.com/keys, then run the cells top to bottom. You'll be prompted to paste the key securely (it is never stored in the notebook).

## 1. Install dependencies

In [ ]:
!pip install -q -U openai-agents ddgs nest_asyncio

## 2. Configure Groq as the model backend

The `openai-agents` SDK talks to any OpenAI-compatible endpoint. Groq exposes one at
`https://api.groq.com/openai/v1`, so we point the SDK's `AsyncOpenAI` client there and
wrap it in an `OpenAIChatCompletionsModel` running **`openai/gpt-oss-120b`**.

We also disable tracing — the SDK's default tracing exporter uploads traces to
OpenAI's platform, which requires an OpenAI API key we don't have here.

In [ ]:
import asyncio
import nest_asyncio
from getpass import getpass

nest_asyncio.apply()  # allows nested event loops, just in case

from agents import (
    Agent,
    Runner,
    function_tool,
    InputGuardrail,
    GuardrailFunctionOutput,
    input_guardrail,
    OpenAIChatCompletionsModel,
    set_tracing_disabled,
)
from openai import AsyncOpenAI
from pydantic import BaseModel
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# --- Get your free Groq API key from https://console.groq.com/keys ---
GROQ_API_KEY = getpass("Enter your Groq API key: ")

# Point the OpenAI SDK client at Groq's OpenAI-compatible endpoint
groq_client = AsyncOpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

# Wrap it so the Agents SDK uses Groq + the open-source gpt-oss-120b model
MODEL_NAME = "openai/gpt-oss-120b"
model = OpenAIChatCompletionsModel(model=MODEL_NAME, openai_client=groq_client)

# Disable OpenAI-platform tracing (we're not using OpenAI's backend)
set_tracing_disabled(True)

print(f"✅ Configured Agents SDK to use Groq model: {MODEL_NAME}")

## 3. Custom tools (order lookup, refunds, and a Groq-friendly web search)

In [ ]:
from ddgs import DDGS

# Simulated order database
ORDERS_DB = {
    "ORD-001": {"item": "Wireless Headphones", "status": "Shipped", "eta": "March 22"},
    "ORD-002": {"item": "Python Programming Book", "status": "Delivered", "eta": "March 18"},
    "ORD-003": {"item": "USB-C Cable 3-pack", "status": "Processing", "eta": "March 25"},
}


@function_tool
def lookup_order(order_id: str) -> str:
    """Look up the status of a customer order by order ID (e.g., ORD-001)."""
    order = ORDERS_DB.get(order_id.upper())
    if order:
        return (
            f"Order {order_id.upper()}:\n"
            f"  Item: {order['item']}\n"
            f"  Status: {order['status']}\n"
            f"  Estimated Arrival: {order['eta']}"
        )
    return f"Order {order_id} not found. Please check the order ID and try again."


@function_tool
def process_refund(order_id: str, reason: str) -> str:
    """Process a refund request for a given order ID with a reason."""
    order = ORDERS_DB.get(order_id.upper())
    if not order:
        return f"Cannot process refund: Order {order_id} not found."
    if order["status"] == "Processing":
        return f"Refund for {order_id} cannot be processed — order hasn't shipped yet. It can be cancelled instead."
    return (
        f"✅ Refund initiated for Order {order_id.upper()}\n"
        f"  Item: {order['item']}\n"
        f"  Reason: {reason}\n"
        f"  Refund amount will be credited within 5-7 business days."
    )


@function_tool
def web_search(query: str) -> str:
    """Search the web for current information (e.g., store policies, product facts)."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=5))
        if not results:
            return f"No web results found for '{query}'."
        formatted = "\n\n".join(
            f"- {r.get('title', 'Untitled')}: {r.get('body', '')} ({r.get('href', '')})"
            for r in results
        )
        return f"Web search results for '{query}':\n\n{formatted}"
    except Exception as e:
        return f"Web search failed: {e}"

print("✅ Tools defined: lookup_order, process_refund, web_search")

## 4. Input guardrail

Same idea as the original: a small classifier agent decides whether the incoming
message is actually a customer-support question. Off-topic messages get blocked
before reaching the triage agent. It now runs on the Groq `gpt-oss-120b` model too.

In [ ]:
class SupportCheck(BaseModel):
    is_support_question: bool
    reasoning: str


guardrail_checker = Agent(
    name="Support Topic Checker",
    instructions="""Determine if the user's message is a customer support question.
    Valid topics: order status, refunds, returns, product questions, shipping, FAQs.
    Invalid topics: personal advice, jokes, coding help, unrelated conversations.
    Return is_support_question=True ONLY for customer support topics.""",
    output_type=SupportCheck,
    model=model,
)


@input_guardrail
async def support_only(ctx, agent, input):
    for attempt in range(3):
        try:
            result = await Runner.run(guardrail_checker, input, context=ctx.context)
            final = result.final_output_as(SupportCheck)
            return GuardrailFunctionOutput(
                output_info={"reasoning": final.reasoning},
                tripwire_triggered=not final.is_support_question,
            )
        except Exception:
            if attempt == 2:
                raise
            await asyncio.sleep(1)

print("✅ Guardrail defined: support_only")

## 5. Specialist agents

In [ ]:
order_agent = Agent(
    name="Order_Status_Agent",
    handoff_description="Handles questions about order status, shipping, and delivery.",
    instructions="""You help customers check their order status.
    Use the lookup_order tool to find order information.
    If the customer doesn't provide an order ID, ask for it.
    Be friendly and professional.""",
    tools=[lookup_order],
    model=model,
)

refund_agent = Agent(
    name="Refund_Agent",
    handoff_description="Handles refund requests, returns, and cancellations.",
    instructions="""You help customers with refunds and returns.
    Use the process_refund tool to initiate refunds.
    Always ask for the order ID and reason before processing.
    Be empathetic and helpful.""",
    tools=[process_refund],
    model=model,
)

faq_agent = Agent(
    name="FAQ_Agent",
    handoff_description="Handles general product questions and frequently asked questions.",
    instructions="""You answer general customer questions and FAQs.
    Use web search when you need current information.
    Common topics: shipping policies, return windows, product details.
    Be helpful and concise.""",
    tools=[web_search],
    model=model,
)

print("✅ Specialist agents defined: Order_Status_Agent, Refund_Agent, FAQ_Agent")

## 6. Triage agent

In [ ]:
triage_agent = Agent(
    name="Customer_Support_Triage",
    instructions="""...""",
    tools=[
        order_agent.as_tool(
            tool_name="handle_order_status",
            tool_description="Handles questions about order status, shipping, and delivery.",
        ),
        refund_agent.as_tool(
            tool_name="handle_refund",
            tool_description="Handles refund requests, returns, and cancellations.",
        ),
        faq_agent.as_tool(
            tool_name="handle_faq",
            tool_description="Handles general product questions and FAQs.",
        ),
    ],
    input_guardrails=[support_only],
    model=model,
)

print("✅ Triage agent defined: Customer_Support_Triage")

## 7. Run the system

In [ ]:
async def handle_customer(message: str):
    """Process a customer message through the support system."""
    print(f"🧑 Customer: {message}")
    try:
        result = await Runner.run(triage_agent, message)
        print(f"🤖 {result.last_agent.name}: {result.final_output}")
    except Exception as e:
        print(f"🚫 Blocked: This doesn't appear to be a support question. ({e})")
    print("=" * 70)
    print()


async def main():
    print("=" * 70)
    print("  CUSTOMER SUPPORT AGENT SYSTEM — GROQ + gpt-oss-120b DEMO")
    print("=" * 70)
    print()

    # Test 1: Order status (→ Order Status Agent → lookup_order tool)
    await handle_customer("Where is my order ORD-001?")

    # Test 2: Refund request (→ Refund Agent → process_refund tool)
    await handle_customer("I want a refund for order ORD-002. The book arrived damaged.")

    # Test 3: General FAQ (→ FAQ Agent → web search)
    await handle_customer("What is Amazon's return policy?")

    # Test 4: Off-topic (→ BLOCKED by guardrail)
    await handle_customer("Can you help me write a poem about cats?")

    print()
    print("🎉 Demo complete! Full support agent system running on Groq's gpt-oss-120b.")
    print()
    print("WHAT THIS PROJECT USED:")
    print("  ✓ Multiple agents with handoffs (triage → specialists)")
    print("  ✓ Custom function tools (lookup_order, process_refund)")
    print("  ✓ Input guardrails (support_only blocks off-topic)")
    print("  ✓ Custom web-search tool (DuckDuckGo, Groq-compatible)")
    print("  ✓ Structured output (SupportCheck for guardrail)")
    print("  ✓ Open-source model (openai/gpt-oss-120b via Groq)")


await main()

## 8. Try your own message

Run this cell as many times as you like with any customer message.

In [ ]:
your_message = "Can you check the status of ORD-003?"  # 👈 edit this
await handle_customer(your_message)